# Notebook 12 — Measurement Jacobians and Innovation

This notebook derives and tests the EKF-SLAM measurement Jacobian and innovation logic.


## Learning objectives

1. Derive the Jacobian $H$ for range-bearing observations.
2. Compute innovation/residual correctly with angle normalization.
3. Build intuition for gating using Mahalanobis distance.
4. Validate analytic Jacobians with finite differences.


In [ ]:
import numpy as np

from notebooks._notebook_utils import set_seed, wrap_angle

SEED = set_seed(12)
np.set_printoptions(precision=5, suppress=True)
print(f"Seed: {SEED}")


## 1) Derive $H$ for range-bearing

For robot pose $x_r=[x,y,\theta]^T$ and landmark $m=[m_x,m_y]^T$:

$$
\Delta x=m_x-x,\quad \Delta y=m_y-y,\quad q=\Delta x^2+\Delta y^2,\quad r=\sqrt{q}.
$$

$$
h(x_r,m)=\begin{bmatrix}r \\ \phi\end{bmatrix}=
\begin{bmatrix}
\sqrt{q} \\
\mathrm{wrap}(\operatorname{atan2}(\Delta y,\Delta x)-\theta)
\end{bmatrix}.
$$

Ignoring wrap for local derivatives:

$$
H_r = \frac{\partial h}{\partial x_r} =
\begin{bmatrix}
-\Delta x/r & -\Delta y/r & 0 \\
\Delta y/q & -\Delta x/q & -1
\end{bmatrix},\;
H_l = \frac{\partial h}{\partial m} =
\begin{bmatrix}
\Delta x/r & \Delta y/r \\
-\Delta y/q & \Delta x/q
\end{bmatrix}.
$$


In [ ]:
def h(robot_pose, landmark):
    dx = landmark[0] - robot_pose[0]
    dy = landmark[1] - robot_pose[1]
    rng = np.hypot(dx, dy)
    bearing = wrap_angle(np.arctan2(dy, dx) - robot_pose[2])
    return np.array([rng, bearing], dtype=float)


def jacobians(robot_pose, landmark, eps=1e-12):
    dx = landmark[0] - robot_pose[0]
    dy = landmark[1] - robot_pose[1]
    q = max(dx*dx + dy*dy, eps)
    r = np.sqrt(q)
    Hr = np.array([[-dx/r, -dy/r, 0.0], [dy/q, -dx/q, -1.0]])
    Hl = np.array([[dx/r, dy/r], [-dy/q, dx/q]])
    return Hr, Hl

robot = np.array([1.2, -0.5, np.deg2rad(25.0)])
landmark = np.array([4.0, 2.0])
Hr, Hl = jacobians(robot, landmark)
print("Hr:\n", Hr)
print("Hl:\n", Hl)


### Numeric check via finite differences


In [ ]:
def finite_diff_jacobian(fun, x, step=1e-6):
    y0 = fun(x)
    J = np.zeros((y0.size, x.size))
    for i in range(x.size):
        xp = x.copy(); xp[i] += step
        xm = x.copy(); xm[i] -= step
        yp = fun(xp)
        ym = fun(xm)
        dy = yp - ym
        dy[1] = wrap_angle(dy[1])
        J[:, i] = dy / (2.0 * step)
    return J

Jr_fd = finite_diff_jacobian(lambda xr: h(xr, landmark), robot)
Jl_fd = finite_diff_jacobian(lambda lm: h(robot, lm), landmark)

print("max |Hr-Hr_fd|:", np.max(np.abs(Hr - Jr_fd)))
print("max |Hl-Hl_fd|:", np.max(np.abs(Hl - Jl_fd)))


## 2) Innovation vector and angle normalization

$$
\nu = z - \hat z,
\qquad \nu_\phi = \mathrm{wrap}(z_\phi - \hat z_\phi).
$$


In [ ]:
def innovation(z_meas, z_pred):
    nu = z_meas - z_pred
    nu[1] = wrap_angle(nu[1])
    return nu

z_pred = np.array([5.0, np.deg2rad(179.0)])
z_meas = np.array([5.1, np.deg2rad(-179.0)])

nu_bad = z_meas - z_pred
nu_good = innovation(z_meas.copy(), z_pred.copy())

print("nu without wrap:", nu_bad, "bearing(deg)=", np.rad2deg(nu_bad[1]))
print("nu with wrap   :", nu_good, "bearing(deg)=", np.rad2deg(nu_good[1]))


## 3) Interactive multi-landmark residuals


In [ ]:
robot = np.array([0.0, 0.0, np.deg2rad(170.0)])
landmarks = np.array([[4.0, 1.0], [3.0, -2.0], [-2.0, 0.2]])

z_pred_list = np.array([h(robot, lm) for lm in landmarks])
noise = np.array([[0.10, np.deg2rad(1.0)], [-0.08, np.deg2rad(-2.5)], [0.05, np.deg2rad(2.0)]])
z_meas_list = z_pred_list + noise
z_meas_list[:,1] = wrap_angle(z_meas_list[:,1])

print("idx | z_pred(range,deg) -> z_meas(range,deg) -> innovation(range,deg)")
for i, (zp, zm) in enumerate(zip(z_pred_list, z_meas_list)):
    nu = innovation(zm.copy(), zp.copy())
    print(
        f"{i:>3d} | ({zp[0]:.3f}, {np.rad2deg(zp[1]):7.2f})"
        f" -> ({zm[0]:.3f}, {np.rad2deg(zm[1]):7.2f})"
        f" -> ({nu[0]:+.3f}, {np.rad2deg(nu[1]):+7.2f})"
    )


## 4) Gating intuition (Mahalanobis distance)

$$
D^2 = \nu^T S^{-1} \nu.
$$

For 2D measurement, a 95% gate is roughly $5.99$.


In [ ]:
R = np.diag([0.15**2, np.deg2rad(3.0)**2])
S = np.array([[0.20**2, 0.0], [0.0, np.deg2rad(5.0)**2]]) + R
Sinv = np.linalg.inv(S)

def maha2(nu):
    return float(nu.T @ Sinv @ nu)

candidate_nus = [
    np.array([0.05, np.deg2rad(1.0)]),
    np.array([0.50, np.deg2rad(12.0)]),
    np.array([0.10, np.deg2rad(-6.0)]),
]
for i, nu in enumerate(candidate_nus):
    print(f"candidate {i}: D^2 = {maha2(nu):.3f}")


## Exercise — wrapped innovation and failure if not wrapped

1. Build a boundary-crossing example (near $\pm\pi$).
2. Compare wrapped/unwrapped innovation.
3. Show gating failure without wrapping.


In [ ]:
# Optional solution
S = np.diag([0.2**2, np.deg2rad(4.0)**2])
Sinv = np.linalg.inv(S)

z_pred = np.array([6.0, np.deg2rad(179.5)])
z_meas = np.array([6.05, np.deg2rad(-179.0)])

nu_bad = z_meas - z_pred
nu_good = innovation(z_meas.copy(), z_pred.copy())

D2_bad = float(nu_bad.T @ Sinv @ nu_bad)
D2_good = float(nu_good.T @ Sinv @ nu_good)

print("nu_bad bearing(deg):", np.rad2deg(nu_bad[1]))
print("nu_good bearing(deg):", np.rad2deg(nu_good[1]))
print("D2 without wrapping:", D2_bad)
print("D2 with wrapping   :", D2_good)


## Recap

- We derived and numerically checked range-bearing Jacobians.
- Angle-wrapped innovation is required for stable EKF updates.
- Gating outcomes can be wrong without wrapping.
